In [1]:
# '''
%pip install openai python-dotenv
%pip install -qU langchain-openai langchain
%pip install langchain_community pyowm
%pip install --upgrade --quiet  youtube_search
#'''
import os
import re
import json
import time
import zipfile
import numpy as np
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

In [2]:
# File setup

REPO_DIR = "HelpHerInvest"
DATA_DIR = os.path.join(REPO_DIR, "Data")

zip_path = os.path.join(DATA_DIR, "final_dataset_20260224v2.csv.zip")
stock_symbols_path = os.path.join(DATA_DIR, "nlp_clean_stock_symbols.csv")
company_data_path = os.path.join(DATA_DIR, "final_dataset_20260224v2.csv")

# Extract zipped CSV 
if not os.path.exists(company_data_path) and os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(DATA_DIR)

print("Stock symbols file exists:", os.path.exists(stock_symbols_path))
print("Company dataset exists:", os.path.exists(company_data_path))

Stock symbols file exists: True
Company dataset exists: True


In [3]:
# Upload files once per session

def upload_user_files(stock_file_path, company_file_path):
    with open(stock_file_path, "rb") as f1:
        stock_file = client.files.create(file=f1, purpose="user_data")
    with open(company_file_path, "rb") as f2:
        company_file = client.files.create(file=f2, purpose="user_data")
    return stock_file.id, company_file.id

stock_file_id, company_file_id = upload_user_files(stock_symbols_path, company_data_path)

print("stock_file_id:", stock_file_id)
print("company_file_id:", company_file_id)

stock_file_id: file-Mj8UyJE59MEUKiYS6EyoLZ
company_file_id: file-FapjdEiE2uTYMyN8um83iZ


In [4]:
# "Training" prompts to use to compare/tune variations

TRAIN_PROMPTS = [
    "I like the fashion industry.",
    "I am interested in renewable energy.",
    "I want healthcare stocks.",
    "I like semiconductor companies.",
    "I am interested in cybersecurity.",
    "I want consumer staples stocks."
]

# "Validation" prompts for final model selection

VALIDATION_PROMPTS = [
    "I want travel and leisure stocks.",
    "I am interested in electric vehicle companies.",
    "I like biotech.",
    "I want cloud computing stocks."
]

print("Training prompts:", len(TRAIN_PROMPTS))
print("Validation prompts:", len(VALIDATION_PROMPTS))

Training prompts: 6
Validation prompts: 4


In [5]:
# Variation 1: baseline prompt

BASELINE_DEVELOPER_PROMPT = """
You are a data scientist specializing in stock market analysis.

For every request, you must use the two files attached in the user message.
The first file is a stock symbol reference dataset.
The second file is a company/fundamental dataset.

Required workflow:
1. Analyze the user's interest.
2. Use the first file to identify 20 relevant stock symbols.
3. Use the second file to rank those 20 symbols.
4. Provide a brief explanation for why the stocks were selected.
5. Provide a justification for the ranking.
6. If the data or user input is insufficient, clearly say so instead of guessing.
"""

# Variations 2 and 3: strict prompt, with and without explicit instruction to comment on data limitations

STRICT_DEVELOPER_PROMPT = """
You are a data scientist specializing in stock market analysis.

You must use the two uploaded files:
1. stock symbol reference dataset
2. company/fundamental dataset

Rules:
- Interpret the user's industry or theme.
- Select exactly 20 relevant stock symbols from the stock symbol file.
- Rank those 20 symbols using the company/fundamental dataset.
- Do not invent missing predictor values.
- If predictor fields are sparse, missing, or insufficient for predictive modeling, explicitly say so.
- If a proper predictive model is not possible, use the most defensible available ranking basis in the data and explain it clearly.
- Be specific and concise.
- Follow the exact output structure below.

Return your answer using EXACTLY these section headings:

Industry Interpretation:
<1 short paragraph>

Selected Symbols:
1. SYMBOL - short reason
2. SYMBOL - short reason
...
20. SYMBOL - short reason

Ranking:
1. SYMBOL - ranking basis
2. SYMBOL - ranking basis
...
20. SYMBOL - ranking basis

Ranking Basis:
<1 short paragraph>

Data Limitations:
<1 short paragraph>
"""

In [6]:
# 3 LLM variations

VARIATIONS = [
    {
        "variation": "V1_baseline_mini",
        "model": "gpt-5-mini",
        "max_output_tokens": 900,
        "developer_prompt": BASELINE_DEVELOPER_PROMPT
    },
    {
        "variation": "V2_strict_mini",
        "model": "gpt-5-mini",
        "max_output_tokens": 1100,
        "developer_prompt": STRICT_DEVELOPER_PROMPT
    },
    {
        "variation": "V3_strict_gpt54",
        "model": "gpt-5.4",
        "max_output_tokens": 1100,
        "developer_prompt": STRICT_DEVELOPER_PROMPT
    }
]

pd.DataFrame(VARIATIONS)[["variation", "model", "max_output_tokens"]]

,variation,model,max_output_tokens
0,V1_baseline_mini,gpt-5-mini,900
1,V2_strict_mini,gpt-5-mini,1100
2,V3_strict_gpt54,gpt-5.4,1100


In [7]:
# LLM inference function

def run_llm_variation(user_input, variation, stock_file_id, company_file_id):
    start_time = time.time()

    response = client.responses.create(
        model=variation["model"],
        max_output_tokens=variation["max_output_tokens"],
        input=[
            {
                "role": "developer",
                "content": [
                    {
                        "type": "input_text",
                        "text": variation["developer_prompt"]
                    }
                ]
            },
            {
                "role": "user",
                "content": [
                    {"type": "input_file", "file_id": stock_file_id},
                    {"type": "input_file", "file_id": company_file_id},
                    {"type": "input_text", "text": user_input}
                ]
            }
        ]
    )

    elapsed = time.time() - start_time
    output_text = response.output_text if hasattr(response, "output_text") else str(response)

    return {
        "raw_output": output_text,
        "latency_seconds": round(elapsed, 2)
    }

In [8]:
# Parsing helpers for evaluation

SECTION_ORDER = [
    "Industry Interpretation",
    "Selected Symbols",
    "Ranking",
    "Ranking Basis",
    "Data Limitations"
]

def extract_section(text, heading):
    """
    Extract text between one section heading and the next.
    """
    idx = SECTION_ORDER.index(heading)
    next_headings = SECTION_ORDER[idx + 1:]

    if next_headings:
        next_pattern = "|".join(re.escape(h) for h in next_headings)
        pattern = rf"^\s*{re.escape(heading)}\s*:\s*(.*?)(?=^\s*(?:{next_pattern})\s*:|\Z)"
    else:
        pattern = rf"^\s*{re.escape(heading)}\s*:\s*(.*)$"

    match = re.search(pattern, text, flags=re.IGNORECASE | re.MULTILINE | re.DOTALL)
    return match.group(1).strip() if match else ""

def count_list_items(section_text):
    """
    Count lines that look like numbered or bulleted list items.
    """
    if not section_text.strip():
        return 0

    lines = section_text.splitlines()
    count = 0
    for line in lines:
        if re.match(r"^\s*(\d+[\.\)]|-|\*)\s+", line.strip()):
            count += 1
    return count

def safe_contains_any(text, keywords):
    text_lower = text.lower()
    return int(any(keyword.lower() in text_lower for keyword in keywords))

In [9]:
# Automatic evaluation metrics

GROUNDING_KEYWORDS = [
    "missing",
    "sparse",
    "insufficient",
    "not enough",
    "unreliable",
    "overfit",
    "fwd_excess",
    "historical",
    "not a forecast",
    "not predictive",
    "data limitations"
]

def score_output(output_text):
    selected_symbols_section = extract_section(output_text, "Selected Symbols")
    ranking_section = extract_section(output_text, "Ranking")
    ranking_basis_section = extract_section(output_text, "Ranking Basis")
    limitations_section = extract_section(output_text, "Data Limitations")

    selected_count = count_list_items(selected_symbols_section)
    ranking_count = count_list_items(ranking_section)

    selected_20_score = 1 if selected_count == 20 else 0
    ranking_20_score = 1 if ranking_count == 20 else 0
    has_ranking_basis = 1 if len(ranking_basis_section) >= 20 else 0
    has_data_limitations = 1 if len(limitations_section) >= 20 else 0
    grounding_score = safe_contains_any(output_text, GROUNDING_KEYWORDS)

    overall_score = np.mean([
        selected_20_score,
        ranking_20_score,
        has_ranking_basis,
        has_data_limitations,
        grounding_score
    ])

    return {
        "selected_count": selected_count,
        "ranking_count": ranking_count,
        "selected_20_score": selected_20_score,
        "ranking_20_score": ranking_20_score,
        "has_ranking_basis": has_ranking_basis,
        "has_data_limitations": has_data_limitations,
        "grounding_score": grounding_score,
        "overall_score": round(float(overall_score), 4)
    }

In [10]:
# Run the models and cache results

CACHE_PATH = Path("week8_llm_results.csv")

def run_experiment(prompt_list, dataset_name, variations, stock_file_id, company_file_id):
    rows = []

    for prompt_id, user_prompt in enumerate(prompt_list, start=1):
        print(f"\nRunning {dataset_name} prompt {prompt_id}: {user_prompt}")

        for variation in variations:
            print(f"  -> {variation['variation']}")

            result = run_llm_variation(
                user_input=user_prompt,
                variation=variation,
                stock_file_id=stock_file_id,
                company_file_id=company_file_id
            )

            metrics = score_output(result["raw_output"])

            row = {
                "dataset": dataset_name,
                "prompt_id": prompt_id,
                "user_prompt": user_prompt,
                "variation": variation["variation"],
                "model": variation["model"],
                "max_output_tokens": variation["max_output_tokens"],
                "latency_seconds": result["latency_seconds"],
                "raw_output": result["raw_output"],
                **metrics
            }

            rows.append(row)
            time.sleep(1)  

    return pd.DataFrame(rows)

# If results already exist, read them instead of re-running
if CACHE_PATH.exists():
    results_df = pd.read_csv(CACHE_PATH)
    print("Loaded cached results from:", CACHE_PATH)
else:
    train_results = run_experiment(
        prompt_list=TRAIN_PROMPTS,
        dataset_name="train",
        variations=VARIATIONS,
        stock_file_id=stock_file_id,
        company_file_id=company_file_id
    )

    validation_results = run_experiment(
        prompt_list=VALIDATION_PROMPTS,
        dataset_name="validation",
        variations=VARIATIONS,
        stock_file_id=stock_file_id,
        company_file_id=company_file_id
    )

    results_df = pd.concat([train_results, validation_results], ignore_index=True)
    results_df.to_csv(CACHE_PATH, index=False)
    print("Saved results to:", CACHE_PATH)

results_df.head(3)


Running train prompt 1: I like the fashion industry.
  -> V1_baseline_mini
  -> V2_strict_mini
  -> V3_strict_gpt54

Running train prompt 2: I am interested in renewable energy.
  -> V1_baseline_mini
  -> V2_strict_mini
  -> V3_strict_gpt54

Running train prompt 3: I want healthcare stocks.
  -> V1_baseline_mini
  -> V2_strict_mini
  -> V3_strict_gpt54

Running train prompt 4: I like semiconductor companies.
  -> V1_baseline_mini
  -> V2_strict_mini
  -> V3_strict_gpt54

Running train prompt 5: I am interested in cybersecurity.
  -> V1_baseline_mini
  -> V2_strict_mini
  -> V3_strict_gpt54

Running train prompt 6: I want consumer staples stocks.
  -> V1_baseline_mini
  -> V2_strict_mini
  -> V3_strict_gpt54

Running validation prompt 1: I want travel and leisure stocks.
  -> V1_baseline_mini
  -> V2_strict_mini
  -> V3_strict_gpt54

Running validation prompt 2: I am interested in electric vehicle companies.
  -> V1_baseline_mini
  -> V2_strict_mini
  -> V3_strict_gpt54

Running valida

,dataset,prompt_id,user_prompt,variation,model,max_output_tokens,latency_seconds,raw_output,selected_count,ranking_count,selected_20_score,ranking_20_score,has_ranking_basis,has_data_limitations,grounding_score,overall_score
0,train,1,I like the fashion industry.,V1_baseline_mini,gpt-5-mini,900,23.27,,0,0,0,0,0,0,0,0.0
1,train,1,I like the fashion industry.,V2_strict_mini,gpt-5-mini,1100,27.58,,0,0,0,0,0,0,0,0.0
2,train,1,I like the fashion industry.,V3_strict_gpt54,gpt-5.4,1100,31.47,Industry Interpretation:\nYou are asking for f...,20,20,1,1,1,1,1,1.0


In [11]:
# Quick inspection

results_df[[
    "dataset",
    "prompt_id",
    "variation",
    "model",
    "latency_seconds",
    "selected_20_score",
    "ranking_20_score",
    "has_ranking_basis",
    "has_data_limitations",
    "grounding_score",
    "overall_score"
]].head(10)

# Example: print one full response
sample_row = results_df.iloc[0]
print("Variation:", sample_row["variation"])
print("Prompt:", sample_row["user_prompt"])
print("-" * 80)
print(sample_row["raw_output"])

Variation: V1_baseline_mini
Prompt: I like the fashion industry.
--------------------------------------------------------------------------------



In [12]:
# Summary metrics

metric_cols = [
    "selected_20_score",
    "ranking_20_score",
    "has_ranking_basis",
    "has_data_limitations",
    "grounding_score",
    "overall_score",
    "latency_seconds"
]

summary_df = (
    results_df
    .groupby(["dataset", "variation", "model"], as_index=False)[metric_cols]
    .mean()
    .round(3)
)

summary_df

,dataset,variation,model,selected_20_score,ranking_20_score,has_ranking_basis,has_data_limitations,grounding_score,overall_score,latency_seconds
0,train,V1_baseline_mini,gpt-5-mini,0.0,0.0,0.0,0.0,0.0,0.0,22.560
1,train,V2_strict_mini,gpt-5-mini,0.0,0.0,0.0,0.0,0.0,0.0,23.615
2,train,V3_strict_gpt54,gpt-5.4,1.0,1.0,1.0,1.0,1.0,1.0,28.077
3,validation,V1_baseline_mini,gpt-5-mini,0.0,0.0,0.0,0.0,0.0,0.0,18.392
4,validation,V2_strict_mini,gpt-5-mini,0.0,0.0,0.0,0.0,0.0,0.0,20.685
5,validation,V3_strict_gpt54,gpt-5.4,1.0,1.0,1.0,1.0,1.0,1.0,26.565


In [13]:
report_table = summary_df.pivot_table(
    index=["variation", "model"],
    columns="dataset",
    values=metric_cols
)

report_table = report_table.round(3)
report_table

grounding_score            has_data_limitations  \
dataset                               train validation                train   
variation        model                                                        
V1_baseline_mini gpt-5-mini             0.0        0.0                  0.0   
V2_strict_mini   gpt-5-mini             0.0        0.0                  0.0   
V3_strict_gpt54  gpt-5.4                1.0        1.0                  1.0   

                                       has_ranking_basis             \
dataset                     validation             train validation   
variation        model                                                
V1_baseline_mini gpt-5-mini        0.0               0.0        0.0   
V2_strict_mini   gpt-5-mini        0.0               0.0        0.0   
V3_strict_gpt54  gpt-5.4           1.0               1.0        1.0   

                            latency_seconds            overall_score  \
dataset                               train validation         train   
variation        model                                                 
V1_baseline_mini gpt-5-mini          22.560     18.392           0.0   
V2_strict_mini   gpt-5-mini          23.615     20.685           0.0   
V3_strict_gpt54  gpt-5.4             28.077     26.565           1.0   

                                       ranking_20_score             \
dataset                     validation            train validation   
variation        model                                               
V1_baseline_mini gpt-5-mini        0.0              0.0        0.0   
V2_strict_mini   gpt-5-mini        0.0              0.0        0.0   
V3_strict_gpt54  gpt-5.4           1.0              1.0        1.0   

                            selected_20_score             
dataset                                 train validation  
variation        model                                    
V1_baseline_mini gpt-5-mini               0.0        0.0  
V2_strict_mini   gpt-5-mini               0.0        0.0  
V3_strict_gpt54  gpt-5.4                  1.0        1.0

In [14]:
report_table_flat = summary_df.copy()
report_table_flat = report_table_flat.sort_values(["variation", "dataset"]).reset_index(drop=True)
report_table_flat

,dataset,variation,model,selected_20_score,ranking_20_score,has_ranking_basis,has_data_limitations,grounding_score,overall_score,latency_seconds
0,train,V1_baseline_mini,gpt-5-mini,0.0,0.0,0.0,0.0,0.0,0.0,22.560
1,validation,V1_baseline_mini,gpt-5-mini,0.0,0.0,0.0,0.0,0.0,0.0,18.392
2,train,V2_strict_mini,gpt-5-mini,0.0,0.0,0.0,0.0,0.0,0.0,23.615
3,validation,V2_strict_mini,gpt-5-mini,0.0,0.0,0.0,0.0,0.0,0.0,20.685
4,train,V3_strict_gpt54,gpt-5.4,1.0,1.0,1.0,1.0,1.0,1.0,28.077
5,validation,V3_strict_gpt54,gpt-5.4,1.0,1.0,1.0,1.0,1.0,1.0,26.565


In [15]:
# Choose the winner using validation only

validation_summary = (
    summary_df[summary_df["dataset"] == "validation"]
    .sort_values(
        by=["overall_score", "grounding_score", "selected_20_score", "ranking_20_score"],
        ascending=False
    )
    .reset_index(drop=True)
)

validation_summary

,dataset,variation,model,selected_20_score,ranking_20_score,has_ranking_basis,has_data_limitations,grounding_score,overall_score,latency_seconds
0,validation,V3_strict_gpt54,gpt-5.4,1.0,1.0,1.0,1.0,1.0,1.0,26.565
1,validation,V1_baseline_mini,gpt-5-mini,0.0,0.0,0.0,0.0,0.0,0.0,18.392
2,validation,V2_strict_mini,gpt-5-mini,0.0,0.0,0.0,0.0,0.0,0.0,20.685
